In [1]:
import numpy as np
import pandas as pd
import json
from pathlib import Path

from jupyter_core.migrate import src
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.linear_model import LogisticRegression

project_root = Path.cwd().resolve().parent.parent
processed_dir = project_root / 'src' / 'data' / 'processed'

# Load flat data, baseline models don't need sequences
X_train = np.load(processed_dir / 'X_train.npy')
X_test  = np.load(processed_dir / 'X_test.npy')
y_train = np.load(processed_dir / 'y_train.npy')
y_test  = np.load(processed_dir / 'y_test.npy')


with open(processed_dir / 'compound_mapping.json') as f:
    compound_mapping = {int(k): v for k, v in json.load(f).items()}

print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_test:  {X_test.shape},  y_test:  {y_test.shape}")

X_train: (1527, 27), y_train: (1527,)
X_test:  (786, 27),  y_test:  (786,)


In [2]:
# Blind classifier baseline, always predicts most frequent class
from collections import Counter

most_common = Counter(y_train).most_common(1)[0][0]
blind_preds = np.full(len(y_test), most_common)

print("Blind Classifier: ")
print(classification_report(y_test, blind_preds, target_names=['HARD', 'MEDIUM', 'SOFT']))
print(f"Accuracy: {accuracy_score(y_test, blind_preds):.4f}")

Blind Classifier: 
              precision    recall  f1-score   support

        HARD       0.41      1.00      0.58       321
      MEDIUM       0.00      0.00      0.00       356
        SOFT       0.00      0.00      0.00       109

    accuracy                           0.41       786
   macro avg       0.14      0.33      0.19       786
weighted avg       0.17      0.41      0.24       786

Accuracy: 0.4084


/Users/stephen/UncertaintyQuantification/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/stephen/UncertaintyQuantification/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/stephen/UncertaintyQuantification/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _war

In [3]:
# Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
rf.fit(X_train, y_train)
rf_preds = rf.predict(X_test)

print("Random Forest: ")
print(classification_report(y_test, rf_preds, target_names=['HARD', 'MEDIUM', 'SOFT']))
print(confusion_matrix(y_test, rf_preds))
print(f"Accuracy: {accuracy_score(y_test, rf_preds):.4f}")

Random Forest: 
              precision    recall  f1-score   support

        HARD       0.50      0.87      0.63       321
      MEDIUM       0.63      0.39      0.48       356
        SOFT       0.50      0.01      0.02       109

    accuracy                           0.53       786
   macro avg       0.54      0.42      0.38       786
weighted avg       0.56      0.53      0.48       786

[[280  41   0]
 [217 138   1]
 [ 68  40   1]]
Accuracy: 0.5331


In [4]:
# Logistic Regression
lr = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
lr.fit(X_train, y_train)
lr_preds = lr.predict(X_test)

print("Logistic Regression: ")
print(classification_report(y_test, lr_preds, target_names=['HARD', 'MEDIUM', 'SOFT']))
print(f"Accuracy: {accuracy_score(y_test, lr_preds):.4f}")

Logistic Regression: 
              precision    recall  f1-score   support

        HARD       0.51      0.49      0.50       321
      MEDIUM       0.58      0.37      0.45       356
        SOFT       0.17      0.38      0.23       109

    accuracy                           0.42       786
   macro avg       0.42      0.41      0.40       786
weighted avg       0.49      0.42      0.44       786

Accuracy: 0.4224


In [5]:
# Feature importance from Random Forest
feature_cols = ['Driver', 'LapNumber', 'Stint', 'Sector1Time', 'Sector2Time', 'Sector3Time', 'SpeedI1', 'SpeedI2', 'SpeedST', 'Compound', 'TyreLife', 'FreshTyre', 'Position', 'IsAccurate', 'AirTemp', 'TrackTemp', 'Rainfall', 'Humidity', 'WindSpeed', 'AvgGapAhead', 'MinGapAhead', 'AvgThrottle', 'AvgBrake', 'AvgSpeed', 'PitLap', 'LapsSinceLastPit', 'GapAheadDelta']

importances = pd.Series(rf.feature_importances_, index=feature_cols)
print("\nFeature Importance: ")
print(importances.sort_values(ascending=False).head(10))


Feature Importance: 
LapNumber           0.066043
LapsSinceLastPit    0.063649
AvgSpeed            0.056533
TrackTemp           0.054112
Sector1Time         0.054025
Sector3Time         0.053185
Humidity            0.049418
SpeedI1             0.048613
SpeedI2             0.047635
Sector2Time         0.047415
dtype: float64


In [9]:
# Summary comparison
results = {
    'Blind Classifier': accuracy_score(y_test, blind_preds),
    'Logistic Regression': accuracy_score(y_test, lr_preds),
    'Random Forest': accuracy_score(y_test, rf_preds),
    'feature_importance' : importances.to_dict(),
}

baseline_dir = project_root / 'src' / 'baseline'
baseline_dir.mkdir(exist_ok=True)

with open(baseline_dir/'baseline.json', 'w') as f:
    json.dump(results, f, indent=2)